# Module 5: Enterprise Integration with MCP and Secure Adapters

This notebook implements a mock **ERP_Connector MCP Server** with safe tools and RBAC controls for SnapAudit.


## 1. Setup and Goals

We want an agent that can read/write to a ledger safely:
- `approve_transaction`
- `flag_fraud`
- RBAC for approvals over $10k (requires `finance_admin` + digital key)


In [23]:
from pathlib import Path
import sys
import json

module_dir = Path.cwd()
if str(module_dir) not in sys.path:
    sys.path.append(str(module_dir))

print("✓ Setup complete")


✓ Setup complete


## 2. Load MCP Adapter Components

Import the secure server, ledger, and client abstractions.


In [24]:
from mcp_secure_adapter import (
    MockERPLedger,
    ERPConnectorMCPServer,
    SnapAuditAgentClient,
    RBACPolicy,
)

print("✓ MCP components imported")


✓ MCP components imported


## 3. Initialize Ledger and Server

Start with a mock ERP ledger and MCP server.


In [25]:
ledger = MockERPLedger()
mcp_server = ERPConnectorMCPServer(ledger)
agent = SnapAuditAgentClient(mcp_server)

print("✓ ERP MCP server initialized")
print(f"Transactions loaded: {len(ledger.transactions)}")


✓ ERP MCP server initialized
Transactions loaded: 4


### Inspect Seed Transactions


In [26]:
ledger.list_transactions()


[{'transaction_id': 'T1001',
  'vendor': 'Staples',
  'amount': '245.50',
  'currency': 'USD',
  'category': 'Office Supplies',
  'status': 'pending',
  'reason': '',
  'updated_at': ''},
 {'transaction_id': 'T1002',
  'vendor': 'Delta Airlines',
  'amount': '1860.00',
  'currency': 'USD',
  'category': 'Travel',
  'status': 'pending',
  'reason': '',
  'updated_at': ''},
 {'transaction_id': 'T1003',
  'vendor': 'Global Consulting',
  'amount': '12500.00',
  'currency': 'USD',
  'category': 'Consulting',
  'status': 'pending',
  'reason': '',
  'updated_at': ''},
 {'transaction_id': 'T1004',
  'vendor': 'AWS',
  'amount': '9800.00',
  'currency': 'USD',
  'category': 'Cloud',
  'status': 'pending',
  'reason': '',
  'updated_at': ''}]

## 4. Tool-by-Tool MCP Testing

Validate each exposed tool directly.


In [27]:
    # Read tool
n = mcp_server.get_transaction("T1002")
print(json.dumps(n, indent=2))


{
  "ok": true,
  "transaction": {
    "transaction_id": "T1002",
    "vendor": "Delta Airlines",
    "amount": "1860.00",
    "currency": "USD",
    "category": "Travel",
    "status": "pending",
    "reason": "",
    "updated_at": ""
  }
}


In [28]:
    # Flag tool
flag_result = mcp_server.flag_fraud(
    transaction_id="T1002",
    actor_role="manager",
    reason="Potential duplicate receipt",
)
print(json.dumps(flag_result, indent=2))


{
  "ok": true,
  "message": "Transaction flagged",
  "transaction": {
    "transaction_id": "T1002",
    "vendor": "Delta Airlines",
    "amount": "1860.00",
    "currency": "USD",
    "category": "Travel",
    "status": "flagged",
    "reason": "Potential duplicate receipt",
    "updated_at": "2026-03-01T15:03:43Z"
  }
}


In [29]:
    # Approve tool under threshold
approve_small = mcp_server.approve_transaction(
    transaction_id="T1001",
    actor_role="agent",
)
print(json.dumps(approve_small, indent=2))


{
  "ok": true,
  "message": "Transaction approved",
  "transaction": {
    "transaction_id": "T1001",
    "vendor": "Staples",
    "amount": "245.50",
    "currency": "USD",
    "category": "Office Supplies",
    "status": "approved",
    "reason": "Policy compliant",
    "updated_at": "2026-03-01T15:03:43Z"
  },
  "policy": "Allowed: amount under threshold"
}


## 5. RBAC Scenarios for High-Value Approvals

For amounts over $10,000:
- `agent` should be denied
- `finance_admin` without key should be denied
- `finance_admin` with correct key should be approved


In [30]:
    # Case A: agent denied over threshold
case_a = mcp_server.approve_transaction("T1003", actor_role="agent")
print("Case A (agent):", case_a.get("ok"), case_a.get("error"))

# Case B: finance_admin without key denied
case_b = mcp_server.approve_transaction("T1003", actor_role="finance_admin")
print("Case B (admin no key):", case_b.get("ok"), case_b.get("error"))

# Case C: finance_admin with key approved
case_c = mcp_server.approve_transaction(
    "T1003",
    actor_role="finance_admin",
    digital_key=RBACPolicy.HIGH_VALUE_KEY,
)
print("Case C (admin + key):", case_c.get("ok"), case_c.get("message"))


Case A (agent): False Denied: only finance_admin can approve transactions over $10k
Case B (admin no key): False Denied: missing/invalid digital key for high-value approval
Case C (admin + key): True Transaction approved


## 6. Agent-Driven End-to-End Flow

Simulate SnapAudit policy output and let the client invoke MCP tools accordingly.


In [31]:
# Policy says compliant -> try approval
flow_1 = agent.process_transaction(
    transaction_id="T1004",
    policy_approved=True,
    actor_role="manager",
)
print("Flow 1 status:", flow_1.get("ok"), flow_1.get("transaction", {}).get("status"), flow_1.get("error"))

# Policy says non-compliant -> flag
flow_2 = agent.process_transaction(
    transaction_id="T1002",
    policy_approved=False,
    actor_role="agent",
)
print("Flow 2 status:", flow_2.get("ok"), flow_2.get("transaction", {}).get("status"), flow_2.get("error"))


Flow 1 status: True approved None
Flow 2 status: True flagged None


## 7. Audit Log Review

Every tool call is logged for compliance tracing.


In [32]:
for row in mcp_server.audit_log:
    print(row)


{'time': '2026-03-01T15:03:43Z', 'tool': 'flag_fraud', 'actor_role': 'manager', 'transaction_id': 'T1002', 'result': 'flagged', 'detail': 'Potential duplicate receipt'}
{'time': '2026-03-01T15:03:43Z', 'tool': 'approve_transaction', 'actor_role': 'agent', 'transaction_id': 'T1001', 'result': 'approved', 'detail': 'Allowed: amount under threshold'}
{'time': '2026-03-01T15:03:44Z', 'tool': 'approve_transaction', 'actor_role': 'agent', 'transaction_id': 'T1003', 'result': 'denied', 'detail': 'Denied: only finance_admin can approve transactions over $10k'}
{'time': '2026-03-01T15:03:44Z', 'tool': 'approve_transaction', 'actor_role': 'finance_admin', 'transaction_id': 'T1003', 'result': 'denied', 'detail': 'Denied: missing/invalid digital key for high-value approval'}
{'time': '2026-03-01T15:03:44Z', 'tool': 'approve_transaction', 'actor_role': 'finance_admin', 'transaction_id': 'T1003', 'result': 'approved', 'detail': 'Allowed: finance_admin with valid digital key'}
{'time': '2026-03-01T15

## 8. Assertions (Smoke Test)


In [33]:
assert case_a["ok"] is False
assert case_b["ok"] is False
assert case_c["ok"] is True
assert flow_2["ok"] is True and flow_2["transaction"]["status"] == "flagged"

print("✓ Module 5 MCP smoke checks passed")


✓ Module 5 MCP smoke checks passed
